# Solution 2.7: Transforming and Merging (Angola IEA and INE trade)

Two sources, two jobs. The survey cleaned in 2.6 becomes analysis ready through
custom functions and `apply`. The INE trade workbooks are then loaded, merged and
stacked.

Every mapping in this notebook is **extracted from a source**, never typed by
hand: variable descriptions come from the SPSS header, country names from the
trade sheet, category names from the workbook that defines them.

**PT:** Duas fontes, dois trabalhos. O inquerito limpo em 2.6 torna-se pronto
para analise com funcoes proprias e `apply`. Depois carregamos, juntamos e
empilhamos os ficheiros de comercio do INE.

Todos os mapeamentos sao **extraidos da fonte**, nunca escritos a mao.

> **Pipeline:** run 2.6 first. Reads `10_cleaned/` and `0_raw/angola`, writes
> `20_processed/`.

### Path Setup (run first)

**PT:** Configuracao dos caminhos.

In [ ]:
import os

import numpy as np
import pandas as pd

DATA_RAW_DIR = '../../data/0_raw/angola'
DATA_CLEAN_DIR = '../../data/10_cleaned'
DATA_PROC_DIR = '../../data/20_processed'

EMPLOYMENT_DIR = 'employment_survey'
TRADE_DIR = 'international_trade'
RAW_FILE = 'IEA_2025_IV_TRIM_IND.sav'

clean_path = os.path.join(DATA_CLEAN_DIR, 'angola_iea_2025q4_clean.csv')
trade_dir = os.path.join(DATA_RAW_DIR, TRADE_DIR)

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Trade workbooks available:')
for name in sorted(os.listdir(trade_dir)):
    print('  ', name)

---

# Part A: the survey

## Task 1: Load the cleaned file and recover its descriptions

CSV keeps no dtypes and no documentation, so both have to be re-established.
The descriptions come back from the SPSS header rather than from a dictionary
typed into this notebook.

**PT:** O CSV nao guarda tipos nem documentacao, por isso ambos tem de ser
restabelecidos. As descricoes voltam do cabecalho SPSS, nao de um dicionario
escrito a mao aqui.

In [ ]:
df = pd.read_csv(clean_path, dtype={'household_id': 'string', 'person_id': 'string',
                                    'cluster_id': 'string'})

# Notebook 2.6 saved the mapping, so no need to reopen the SPSS header
# O caderno 2.6 gravou o mapeamento, nao e preciso reabrir o cabecalho SPSS
codebook_path = os.path.join(DATA_CLEAN_DIR, 'angola_iea_2025q4_codebook.csv')
codebook_df = pd.read_csv(codebook_path)
descriptions = dict(zip(codebook_df['new_name'], codebook_df['description']))

print('Survey:', df.shape)
print('Descriptions recovered:', len(descriptions))
print()
for col in ['worked_for_pay', 'available_last_week', 'sought_job']:
    print(f'{col:24s} {str(descriptions[col])[:58]}')

**Answers:**

- 53,353 rows and 27 columns, and a description for each.
- The descriptions come from the file notebook 2.6 wrote, so the readable
  names still lead back to the original questionnaire wording.
- `worked_for_pay` asks whether the person worked at least one hour for pay in
  the last seven days, the ILO employment criterion, which is why Task 3 uses it.

---

## Task 2: A custom function on one column

`apply` on a Series runs your function once per value. Use it when the rule needs
branching that a vectorised expression cannot express clearly.

The bands are not arbitrary: 15 is the ILO working age threshold and 65 the usual
retirement reference, so the function encodes a definition rather than a
convenience.

**PT:** `apply` numa Serie corre a funcao para cada valor. Use quando a regra tem
ramificacoes. As faixas nao sao arbitrarias: 15 anos e o limiar de idade ativa da
OIT e 65 a referencia de reforma.

In [ ]:
def age_band(age):
    """ILO oriented age band for one person."""
    if pd.isna(age):
        return 'Unknown'
    if age < 15:
        return 'Child'
    if age < 25:
        return 'Youth'
    if age < 65:
        return 'Adult'
    return 'Elderly'


df['age_band'] = df['age'].apply(age_band)
df['age_band'].value_counts()

**Answers:**

- Child 23,528, Youth 10,901, Adult 17,239, Elderly 1,685.
- Children are 44% of the sample, which is the single most important fact about
  Angola's labour market and is visible before any modelling.
- A vectorised `pd.cut` would do the same job faster. The function wins when the
  rule needs a guard (`pd.isna`) and a name that documents the definition.

---

## Task 3: A custom function across several columns

`apply(axis=1)` passes a whole row, so the function can read many columns at
once. Labour force status is the natural case: it depends on eight answers and no
single-column expression can express it.

The values are Portuguese labels, not codes, because 2.5 loaded the file with
`convert_categoricals=True`. The function reads almost like the questionnaire.

**PT:** `apply(axis=1)` passa a linha inteira, por isso a funcao pode ler varias
colunas. A situacao perante o trabalho depende de oito respostas. Os valores sao
etiquetas em portugues, por isso a funcao le-se quase como o questionario.

In [ ]:
def labour_force_status(row):
    """ILO status for one person, from the survey's own answers.

    Employed: worked for pay or profit, or has a job they were absent from.
    Unemployed: not employed, looked for work, and available to start.
    """
    if row['age'] < 15:
        return 'Outside labour force'

    worked = (row['worked_for_pay'], row['worked_own_account'], row['absent_from_job'])
    if 'Sim' in worked:
        return 'Employed'

    searched = 'Sim' in (row['sought_job'], row['sought_business'])
    available = 'Sim' in (row['available_last_week'], row['available_next_2weeks'])
    if searched and available:
        return 'Unemployed'

    return 'Outside labour force'


df['lf_status'] = df.apply(labour_force_status, axis=1)
df['lf_status'].value_counts()

**Answers:**

- 15,745 employed, 2,633 unemployed, 34,975 outside the labour force.
- Availability tests `available_last_week` **or** `available_next_2weeks` because `available_next_2weeks` is only asked of
  people who answered no to `available_last_week`. Testing `available_next_2weeks` alone would discard
  almost every unemployed person, since it is 98.5% missing by design.
- `'Sim' in (...)` works on the labels directly. Had we loaded raw codes we would
  be comparing to `1` and `2` and would need the codebook open to read the
  function at all.
- This is slow: `apply(axis=1)` walks 53,345 rows in Python. It is the right
  trade here because the rule is a definition somebody must be able to audit.

---

## Task 4: Weight the result

Each person represents many Angolans, and `weight` says how
many. An unweighted rate describes the sample; a weighted rate describes the
country.

**PT:** Cada pessoa representa muitos angolanos, e o ponderador diz quantos. Uma
taxa nao ponderada descreve a amostra; uma taxa ponderada descreve o pais.

In [ ]:
def weighted_share(mask, weights):
    """Share of the weighted population selected by `mask`, as a percentage."""
    return weights[mask].sum() / weights.sum() * 100


weight = df['weight']
in_labour_force = df['lf_status'].isin(['Employed', 'Unemployed'])

unemployment = weighted_share(df['lf_status'] == 'Unemployed', weight[in_labour_force])
participation = weighted_share(in_labour_force, weight[df['age'] >= 15])

print(f'Unemployment rate:  {unemployment:5.1f}%')
print(f'Participation rate: {participation:5.1f}%')

**Answers:**

- Unemployment 14.5%, participation 62.8%.
- This is the strict definition: it counts only people who actively searched. A
  relaxed measure that also counts those who want work but stopped looking runs
  far higher, and INE's published headline is closer to that. Always say which
  one a table reports.
- The weights matter most for totals. Here they barely move the rate, because
  unemployment happens to be spread evenly across the weighting strata, which is
  luck rather than a reason to skip them.

---

## Task 5: Household size without `groupby`

The file has a household size column, but it is empty in every row, so 2.5 never
loaded it. Rebuild it from the roster: count the rows sharing each `household_id`, then
map that count back onto every person.

**PT:** O ficheiro tem uma coluna de dimensao do agregado, mas esta vazia, por
isso 2.5 nao a carregou. Reconstrua contando as linhas por `household_id`.

In [ ]:
df['hh_size'] = df['household_id'].map(df['household_id'].value_counts())

per_person = df['hh_size'].mean()
per_household = df.drop_duplicates('household_id')['hh_size'].mean()

print(f'Mean household size per person:    {per_person:.2f}')
print(f'Mean household size per household: {per_household:.2f}')

**Answers:**

- 5.58 per person, 4.09 per household. Both are correct and they answer different
  questions.
- A household of 10 contributes 10 rows and a household of 1 contributes one, so
  averaging over rows over-weights large households. "The average person lives in
  a household of 5.58" and "the average household has 4.09 people" are both true.
- Publishing the per person figure as "average household size" is a classic
  error. Deduplicate to the household before averaging a household attribute.

---

# Part B: the trade workbooks

## Task 6: One loader for seven workbooks

INE publishes seven workbooks with the same human readable layout: two title
rows, the real header, a blank row, a `Total Geral` row, the data, and a source
footer. Writing the cleanup once and calling it repeatedly is the whole reason to
define a function.

Rows are dropped by a **property** (`Descrição` is empty) rather than by
position, so the loader survives INE adding a line next quarter.

**PT:** O INE publica sete ficheiros com o mesmo formato: duas linhas de titulo,
o cabecalho, uma linha vazia, o `Total Geral`, os dados, e o rodape da fonte.
Escrever a limpeza uma vez justifica definir uma funcao. As linhas sao removidas
por uma propriedade, nao por posicao.

In [ ]:
def load_trade_sheet(file_name, sheet_name, label_col):
    """Load one INE trade sheet, stripped of title, total and footer rows.

    `label_col` is the descriptive column, which is empty on exactly the rows we
    do not want.
    """
    path = os.path.join(trade_dir, file_name)
    frame = pd.read_excel(path, sheet_name=sheet_name, skiprows=2)
    frame.columns = frame.columns.str.replace('\n', ' ', regex=False).str.strip()
    frame = frame[frame[label_col].notna()].copy()
    return frame


PARTNERS = 'Comercio Externo de Bens por Países Parceiros.xlsx'
exports = load_trade_sheet(PARTNERS, 'Exportação por Países (USD)', 'País')
imports = load_trade_sheet(PARTNERS, 'Importação por Países (USD)', 'País')

print('exports:', exports.shape, '| imports:', imports.shape)
exports[['Código', 'País', 'Ano 2025']].head()

**Answers:**

- 249 rows each: 248 partner countries plus `ZZ`, Desconhecido, meaning the
  partner was not recorded.
- The single `notna` filter removes the blank row, the `Total Geral` row and the
  `Fonte: INE` footer together, because none of them carries a country name.
- The values are in **thousands of US dollars**, stated in the sheet's own second
  line. Nothing in the column names says so, which is exactly how unit errors
  reach publication.

---

## Task 7: Extract the code to name mapping

The country names live in the sheet. Build the lookup from there rather than
typing 248 names, and it stays correct when INE revises one.

**PT:** Os nomes dos paises estao na propria folha. Construa a correspondencia a
partir dela em vez de escrever 248 nomes a mao.

In [ ]:
country_names = dict(zip(exports['Código'], exports['País']))

print('Countries mapped:', len(country_names))
print('Sample:', {k: country_names[k] for k in list(country_names)[:4]})
print('ZZ means:', country_names['ZZ'])

**Answers:**

- 249 entries, one per row of the sheet.
- `ZZ` is Desconhecido, unknown. It is not an error to delete: it is a quantified
  gap in the trade statistics and belongs in a footnote, because dropping it
  would make the export total wrong.

---

## Task 8: Merge exports against imports, then classify with `apply(axis=1)`

Both tables carry one row per country, so this is a one to one merge and
`validate` should say so.

The classification then needs both columns at once, which is the second natural
use of `apply(axis=1)`.

**PT:** As duas tabelas tem uma linha por pais, por isso a juncao e um para um.
A classificacao precisa das duas colunas ao mesmo tempo, o segundo uso natural de
`apply(axis=1)`.

In [ ]:
YEAR = 'Ano 2025'

trade = pd.merge(
    exports[['Código', 'País', YEAR]].rename(columns={YEAR: 'exports_thousand_usd'}),
    imports[['Código', YEAR]].rename(columns={YEAR: 'imports_thousand_usd'}),
    on='Código', how='outer', indicator=True, validate='one_to_one',
).rename(columns={'Código': 'country_code', 'País': 'country_name'})

print(trade['_merge'].value_counts())
trade = trade.drop(columns='_merge')
print('Merged:', trade.shape)

In [ ]:
def partner_profile(row):
    """Describe Angola's 2025 relationship with one partner.

    Reads both flows, so it has to run on the row rather than on a column.
    """
    sold = row['exports_thousand_usd']
    bought = row['imports_thousand_usd']

    if pd.isna(sold) or pd.isna(bought):
        return 'Incomplete'
    if sold + bought < 1000:
        return 'Negligible'
    if sold > 2 * bought:
        return 'Angola mainly sells'
    if bought > 2 * sold:
        return 'Angola mainly buys'
    return 'Two way'


trade['balance_thousand_usd'] = (trade['exports_thousand_usd'].fillna(0)
                                 - trade['imports_thousand_usd'].fillna(0))
trade['profile'] = trade.apply(partner_profile, axis=1)

print(trade['profile'].value_counts())

In [ ]:
print('Largest surpluses:')
print(trade.nlargest(5, 'balance_thousand_usd')[
    ['country_name', 'exports_thousand_usd', 'imports_thousand_usd', 'profile']
].to_string(index=False))
print()
print('Largest deficits:')
print(trade.nsmallest(5, 'balance_thousand_usd')[
    ['country_name', 'exports_thousand_usd', 'imports_thousand_usd', 'profile']
].to_string(index=False))

**Answers:**

- All 249 partners are `both`, and `validate='one_to_one'` passed, so each
  appears exactly once on each side. Confirming a clean merge is a result.
- China is by far the largest surplus partner and Portugal the largest deficit,
  which matches Angola's oil export and consumer goods import profile.
- The `Negligible` threshold is a judgement, not a fact. State it: without it the
  ratio rules would label a partner with 3 thousand dollars of trade as
  "mainly sells", which is technically true and useless.
- `fillna(0)` before subtracting treats "no trade recorded" as zero trade. That is
  reasonable here and would not be if the gap meant "not yet reported".

---

## Task 9: A hierarchy that will double count if you let it

The economic categories workbook (CGCE) codes a tree in the length of the code:
`1` is a section, `11` a group inside it, `111` a subgroup inside that. Summing
the column adds every level together.

A one line function applied to the code column exposes the structure, and the
published `Total Geral` is the check.

**PT:** O ficheiro de Grandes Categorias Economicas codifica uma arvore no
comprimento do codigo: `1` seccao, `11` grupo, `111` subgrupo. Somar a coluna
soma todos os niveis. O `Total Geral` publicado serve de verificacao.

In [ ]:
CGCE_FILE = 'Comercio Externo de Bens por Grandes Categorias Económicas.xlsx'
cgce = load_trade_sheet(CGCE_FILE, 'Export Cat. Económica (USD)', 'Descrição')
cgce['CGCE'] = cgce['CGCE'].astype('string')

published_total = cgce.loc[cgce['Descrição'] == 'Total Geral', YEAR].iloc[0]
cgce = cgce[cgce['CGCE'].notna()].copy()

print('Published Total Geral:', f'{published_total:,.0f}')
print('Category rows:', len(cgce))
cgce[['CGCE', 'Descrição', YEAR]].head()

In [ ]:
def cgce_level(code):
    """Depth in the CGCE tree: 1 section, 2 group, 3 subgroup."""
    if pd.isna(code):
        return 0
    return len(str(code).strip())


cgce['level'] = cgce['CGCE'].apply(cgce_level)
print(cgce['level'].value_counts().sort_index())

In [ ]:
naive = cgce[YEAR].sum()
sections_only = cgce.loc[cgce['level'] == 1, YEAR].sum()

print(f'Published total:        {published_total:15,.0f}')
print(f'Sum of every row:       {naive:15,.0f}  <- {naive / published_total:.2f}x')
print(f'Sum of level 1 only:    {sections_only:15,.0f}')
print()
print('Level 1 matches the published total:',
      bool(abs(sections_only - published_total) < 1))

**Answers:**

- 7 sections, 14 groups, 6 subgroups.
- Summing every row gives **exactly twice** the published total, because each
  section already contains its groups and each group its subgroups. The 2.00x is
  not a coincidence: every value is counted once at its own level and again in
  every ancestor.
- Filtering to level 1 reproduces INE's `Total Geral` to the unit, which is how
  you know the rule is right rather than merely plausible.
- This is why the processing step needs justifying. Nothing in the file warns
  you, no error is raised, and a chart built on the naive sum is silently double
  the truth.

---

## Task 10: Stack the two flows

Merging combines columns, appending combines rows. Exports and imports have the
same shape, so a long format with a `flow` column is often easier to chart and
group than two wide columns.

**PT:** Juntar combina colunas, empilhar combina linhas. Exportacoes e
importacoes tem a mesma forma, por isso o formato longo com uma coluna `flow` e
mais facil de usar.

In [ ]:
long_exports = exports[['Código', 'País', YEAR]].assign(flow='Export')
long_imports = imports[['Código', 'País', YEAR]].assign(flow='Import')

flows = pd.concat([long_exports, long_imports], ignore_index=True)
flows = flows.rename(columns={'Código': 'country_code', 'País': 'country_name',
                              YEAR: 'value_thousand_usd'})

print('Stacked:', flows.shape)
print(flows['flow'].value_counts())
print()
print(flows.groupby('flow')['value_thousand_usd'].sum().round(0))

**Answers:**

- 498 rows, 249 of each flow, and no column is empty because both inputs had
  identical column names before the concat.
- The `flow` column is added **before** stacking. Without it the two halves are
  indistinguishable and the append is irreversible.
- Angola exports far more than it imports in 2025, which is the trade surplus the
  balance column showed country by country.

---

## Task 11: Save

**PT:** Gravar os resultados.

In [ ]:
os.makedirs(DATA_PROC_DIR, exist_ok=True)

survey_out = os.path.join(DATA_PROC_DIR, 'angola_iea_2025q4_features.csv')
trade_out = os.path.join(DATA_PROC_DIR, 'angola_trade_partners.csv')
flows_out = os.path.join(DATA_PROC_DIR, 'angola_trade_flows.csv')

df.to_csv(survey_out, index=False)
trade.to_csv(trade_out, index=False)
flows.to_csv(flows_out, index=False)

print('survey:', df.shape, '| trade:', trade.shape, '| flows:', flows.shape)

**Answers:**

- Three files: the survey with its derived columns, the partner balance table,
  and the long format flows.
- The survey gained `age_band`, `lf_status` and `hh_size`, all three produced by
  functions somebody can read and argue with, which is the point when a number
  ends up in a publication.
- Nothing was written into `0_raw/`.